In [12]:
from flask import Flask, request, jsonify, render_template_string, render_template
import tensorflow as tf
import numpy as np
import os
import requests
import numpy as np
from flask import Flask, request, render_template_string, jsonify
import numpy as np
import pickle

In [13]:
import re
from dataclasses import dataclass, asdict
from email import policy
from email.parser import BytesParser
from email.utils import parseaddr

In [ ]:
app = Flask(__name__)
with open(r"C:\onedrive\Desktop\Senior Proejct\Project\tfidf_vectorizer.pkl", "rb") as f:
    vectorizer = pickle.load(f)
# app = Flask(__name__)
# with open(r"C:\Users\jason\Downloads\COMP-490-Phishing-Detection-System-main\tfidf_vectorizer.pkl", "rb") as f:
#     vectorizer = pickle.load(f)

In [15]:

# Load the model
MODEL_PATH = r"phishing_model.keras"
if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f"Model not found at {MODEL_PATH}")
model = tf.keras.models.load_model(MODEL_PATH)




In [16]:
#To have the home page link to the new html pages
@app.route("/about")
def about(): 
    return render_template("about.html")
@app.route("/docs")
def docs(): 
    return render_template("documentation.html")
@app.route("/security-tips")
def security_tips(): 
    return render_template("tips.html")

In [17]:
# -----------------------------
# Header-based phishing signals
# -----------------------------

@dataclass
class Reason:
    id: str
    severity: str  # LOW/MED/HIGH
    message: str
    evidence: dict

def _safe_lower(s):
    return (s or "").strip().lower()

def extract_domain(addr: str) -> str:
    _, email_addr = parseaddr(addr or "")
    email_addr = email_addr.strip()
    if "@" not in email_addr:
        return ""
    return email_addr.split("@", 1)[1].lower().strip()

def extract_display_name(addr: str) -> str:
    name, _ = parseaddr(addr or "")
    return (name or "").strip()

def clamp01(x: float) -> float:
    return 0.0 if x < 0 else 1.0 if x > 1.0 else x

def score_to_tier(score: float) -> str:
    if score >= 0.75:
        return "HIGH"
    if score >= 0.40:
        return "MED"
    return "LOW"

def parse_auth_results_blob(blob: str) -> dict:
    """
    Heuristic parse for common header strings like:
      spf=pass|fail...
      dkim=pass|fail...
      dmarc=pass|fail...
      header.from=example.com
      smtp.mailfrom=someone@example.com
    """
    b = _safe_lower(blob)
    out = {}

    def find_result(token: str) -> str:
        m = re.search(rf"\b{token}\s*=\s*([a-z0-9_-]+)", b)
        return m.group(1) if m else ""

    for k in ("spf", "dkim", "dmarc"):
        r = find_result(k)
        if r:
            out[k] = r

    m_from = re.search(r"\bheader\.from\s*=\s*([a-z0-9.\-]+)", b)
    if m_from:
        out["header.from"] = m_from.group(1)

    m_mailfrom = re.search(r"\b(smtp\.mailfrom|mailfrom)\s*=\s*([a-z0-9._%+\-]+@[a-z0-9.\-]+)", b)
    if m_mailfrom:
        out["smtp.mailfrom"] = m_mailfrom.group(2)

    return out

def normalize_auth_result(r: str) -> str:
    r = _safe_lower(r)
    mapping = {"bestguesspass": "pass"}
    return mapping.get(r, r)

def domains_mismatch(d1: str, d2: str) -> bool:
    d1, d2 = _safe_lower(d1), _safe_lower(d2)
    return bool(d1) and bool(d2) and d1 != d2

def is_external(domain: str, internal_domains: list[str]) -> bool:
    d = _safe_lower(domain)
    return bool(d) and all(not d.endswith(idom) for idom in internal_domains)

def compute_header_risk_from_raw_text(
    raw_email_text: str,
    internal_domains: list[str] | None = None,
    exec_names: list[str] | None = None,
) -> dict:
    """
    Accepts a pasted raw email string.
    Tries to parse headers with email.parser. If parsing fails, falls back to simple regex extraction.
    Returns dict: score, tier, reasons[], extracted{}
    """
    internal_domains = [d.lower().strip() for d in (internal_domains or []) if d.strip()]
    exec_names = [n.strip() for n in (exec_names or []) if n.strip()]

    # reasonable defaults (EDIT these)
    if not internal_domains:
        internal_domains = ["company.com"]

    # Try to parse using BytesParser
    raw_bytes = (raw_email_text or "").encode("utf-8", errors="replace")
    try:
        msg = BytesParser(policy=policy.default).parsebytes(raw_bytes)

        from_h = (msg.get("From") or "").strip()
        reply_to_h = (msg.get("Reply-To") or "").strip()
        return_path_h = (msg.get("Return-Path") or "").strip()

        # gather auth-related headers
        auth_lines = []
        for hn in ["Authentication-Results", "ARC-Authentication-Results", "Received-SPF"]:
            for v in msg.get_all(hn, []):
                if isinstance(v, str) and v.strip():
                    auth_lines.append(v.strip())
        auth_blob = "\n".join(auth_lines)

    except Exception:
        # Fallback regex extraction if it's not a full RFC822 email
        def grab(name: str) -> str:
            m = re.search(rf"^{re.escape(name)}:\s*(.+)$", raw_email_text, flags=re.MULTILINE | re.IGNORECASE)
            return m.group(1).strip() if m else ""

        from_h = grab("From")
        reply_to_h = grab("Reply-To")
        return_path_h = grab("Return-Path")
        # auth results are often long, grab a few possible lines
        auth_blob = "\n".join([
            grab("Authentication-Results"),
            grab("ARC-Authentication-Results"),
            grab("Received-SPF"),
        ]).strip()

    from_domain = extract_domain(from_h)
    reply_domain = extract_domain(reply_to_h) if reply_to_h else ""
    return_path_domain = extract_domain(return_path_h)

    display_name = extract_display_name(from_h)
    display_name_norm = _safe_lower(display_name)

    auth = parse_auth_results_blob(auth_blob)
    spf = normalize_auth_result(auth.get("spf", ""))
    dkim = normalize_auth_result(auth.get("dkim", ""))
    dmarc = normalize_auth_result(auth.get("dmarc", ""))

    score = 0.0
    reasons: list[Reason] = []

    def add(rid: str, sev: str, message: str, evidence: dict, delta: float):
        nonlocal score
        reasons.append(Reason(rid, sev, message, evidence))
        score += delta

    # DMARC
    if dmarc in ("fail", "permerror"):
        add("DMARC_FAIL", "HIGH", "DMARC failed for this message.",
            {"dmarc": dmarc, "Authentication-Results": auth_blob[:500]}, 0.45)
    elif dmarc and dmarc not in ("pass", "none"):
        add("DMARC_SUSPICIOUS", "MED", f"DMARC is {dmarc}.",
            {"dmarc": dmarc, "Authentication-Results": auth_blob[:500]}, 0.20)

    # SPF
    if spf in ("fail", "softfail", "permerror"):
        add("SPF_FAIL", "MED" if spf == "softfail" else "HIGH",
            f"SPF is {spf}.", {"spf": spf, "Authentication-Results": auth_blob[:500]},
            0.25 if spf == "softfail" else 0.35)

    # DKIM
    if dkim in ("fail", "permerror"):
        add("DKIM_FAIL", "HIGH", f"DKIM is {dkim}.",
            {"dkim": dkim, "Authentication-Results": auth_blob[:500]}, 0.30)

    # From vs smtp.mailfrom mismatch (heuristic)
    header_from_dom = _safe_lower(auth.get("header.from", "")) or from_domain
    smtp_mailfrom_dom = extract_domain(auth.get("smtp.mailfrom", ""))
    if header_from_dom and smtp_mailfrom_dom and domains_mismatch(header_from_dom, smtp_mailfrom_dom):
        add("FROM_MAILFROM_MISMATCH", "MED",
            "Header From domain differs from SMTP MAIL FROM domain (possible spoofing).",
            {"header.from": header_from_dom, "smtp.mailfrom": auth.get("smtp.mailfrom", "")},
            0.18)

    # Reply-To mismatch
    if reply_domain and from_domain and domains_mismatch(reply_domain, from_domain):
        add("REPLY_TO_MISMATCH", "HIGH",
            "Reply-To domain differs from From domain.",
            {"From": from_h, "Reply-To": reply_to_h},
            0.35)

    # Display-name spoofing (only triggers if you supply exec_names)
    exec_norm = {_safe_lower(n) for n in exec_names if n.strip()}
    if exec_norm and display_name_norm and display_name_norm in exec_norm and is_external(from_domain, internal_domains):
        add("DISPLAY_NAME_SPOOF", "HIGH",
            "Sender display name matches a trusted/internal executive, but the sender domain is external.",
            {"Display-Name": display_name, "From": from_h, "From-Domain": from_domain},
            0.40)

    # Return-Path mismatch (often legit; lower weight)
    if return_path_domain and from_domain and domains_mismatch(return_path_domain, from_domain):
        add("RETURN_PATH_MISMATCH", "MED",
            "Return-Path domain differs from From domain (can be legit, but is a common spoofing signal).",
            {"From": from_h, "Return-Path": return_path_h},
            0.15)

    # small bonus if all pass
    if dmarc == "pass" and spf == "pass" and dkim == "pass":
        score -= 0.10

    score = clamp01(score)

    extracted = {
        "From": from_h,
        "From-Domain": from_domain,
        "Reply-To": reply_to_h,
        "Reply-To-Domain": reply_domain,
        "Return-Path": return_path_h,
        "Return-Path-Domain": return_path_domain,
        "Display-Name": display_name,
        "spf": spf,
        "dkim": dkim,
        "dmarc": dmarc,
    }

    return {
        "header_risk_score": score,
        "tier": score_to_tier(score),
        "reasons": [asdict(r) for r in reasons],
        "extracted": extracted,
    }


# -----------------------------
# HTML (unchanged except it expects header_risk + prediction vars)
# -----------------------------
HOME_HTML = """
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <title>Phishing Detection System </title>

    <style>
        @import url('https://fonts.googleapis.com/css2?family=Orbitron:wght@400;600;700&family=Roboto+Mono:wght@300;400;700&display=swap');

        body {
            margin: 0;
            background: #0a0a0f;
            font-family: "Roboto Mono", monospace;
            color: #f6f6f6;
            overflow-x: hidden;
        }

        .noise {
            pointer-events: none;
            position: fixed;
            width: 100%;
            height: 100%;
            top: 0; left: 0;
            background-image: url('https://i.imgur.com/o9bV8Zf.png');
            opacity: 0.05;
            z-index: 9999;
        }

        .scanlines {
            pointer-events: none;
            position: fixed;
            width: 100%;
            height: 100%;
            top: 0; left: 0;
            background: repeating-linear-gradient(
                to bottom,
                rgba(255,255,255,0.03),
                rgba(255,255,255,0.03) 2px,
                transparent 2px,
                transparent 4px
            );
            z-index: 9998;
        }

        nav {
            background: #fcee09;
            color: #000;
            padding: 18px 40px;
            font-family: Orbitron, sans-serif;
            font-weight: 700;
            border-bottom: 4px solid cyan;
            display: flex;
            justify-content: space-between;
            letter-spacing: 2px;
            clip-path: polygon(0 0, 96% 0, 100% 100%, 0 100%);
        }

        nav ul {
            list-style: none;
            display: flex;
            gap: 35px;
            margin: 0;
        }
        nav li {
            cursor: pointer;
            transition: 0.2s;
        }
        nav li:hover {
            color: #ff006e;
            text-shadow: 0 0 6px #ff006e, 0 0 12px #ff006e;
        }

        h1.glitch {
            font-family: Orbitron, sans-serif;
            font-size: 52px;
            color: #fcee09;
            text-align: center;
            margin-top: 100px;
            font-weight: 700;
            position: relative;
        }

        h1.glitch::before,
        h1.glitch::after {
            content: "PHISHING DETECTION SYSTEM";
            position: absolute;
            left: 0;
            top: 0;
            width: 100%;
        }

        h1.glitch::before {
            color: #00eaff;
            transform: translate(3px, 0);
            opacity: 0.6;
        }

        h1.glitch::after {
            color: #ff006e;
            transform: translate(-3px, 0);
            opacity: 0.6;
        }

        .subtitle {
            text-align: center;
            color: #00eaff;
            font-size: 18px;
            margin-top: 10px;
            letter-spacing: 1px;
        }

        .box {
            background: rgba(15, 15, 25, 0.9);
            width: 650px;
            margin: 60px auto;
            padding: 40px;
            border: 2px solid #fcee09;
            box-shadow: 0 0 15px #fcee09;
            border-radius: 0;
            clip-path: polygon(0 0, 100% 0, 100% 92%, 95% 100%, 0 100%);
            position: relative;
        }

        .box::before {
            content: "";
            position: absolute;
            right: -4px;
            top: 10px;
            height: 40%;
            width: 4px;
            background: #ff006e;
            box-shadow: 0 0 10px #ff006e;
        }

        .box h2 {
            color: #fcee09;
            font-family: Orbitron;
        }

        textarea {
            width: 100%;
            height: 180px;
            background: #050505;
            color: #fcee09;
            border: 2px solid #00eaff;
            border-radius: 0;
            padding: 10px;
            font-size: 14px;
            transition: 0.2s;
            outline: none;
        }

        textarea:focus {
            box-shadow: 0 0 12px #00eaff;
        }

        button {
            margin-top: 15px;
            width: 100%;
            padding: 14px;
            background: #ff006e;
            border: none;
            color: white;
            font-weight: bold;
            font-size: 16px;
            cursor: pointer;
            font-family: Orbitron;
            letter-spacing: 2px;
            transition: 0.3s;
        }

        button:hover {
            background: #fcee09;
            color: #000;
            box-shadow: 0 0 12px #fcee09;
        }

        .result {
            margin-top: 25px;
            background: #111;
            padding: 15px;
            border-left: 4px solid #fcee09;
            color: #fcee09;
            box-shadow: 0 0 12px #fcee09 inset;
            font-size: 15px;
        }
    </style>
</head>

<body>

<div class="noise"></div>
<div class="scanlines"></div>

<nav>
    <div>PHISHING DETECTION SYSTEM by Coated Goders</div>
    <ul>
        <li><a href="/">Home</a></li>
        <li><a href="/about">About</a></li>
        <li><a href="/docs">Docs</a></li>
        <li><a href="/security-tips">Security Tips</a></li>
    </ul>
</nav>

<h1 class="glitch">PHISHING DETECTION SYSTEM</h1>
<p class="subtitle">Machine Learning Based Email Scanner for Phising Prevention</p>

<div class="box">
    <h2>Email Vector Analyzer</h2>
    <p>Input transmission:</p>

    <form action="/predict" method="post">
        <textarea name="email_text" placeholder="Paste raw email content..."></textarea>
        <button type="submit">EXECUTE SCAN</button>
    </form>

    {% if header_risk %}
    <div class="result" style="margin-top:15px; border-left-color:#00eaff; color:#00eaff;">
        ▸ Header Risk Tier: {{ header_risk.tier }} <br>
        ▸ Header Risk Score: {{ header_risk.header_risk_score | round(3) }} <br><br>

        {% if header_risk.reasons and header_risk.reasons|length > 0 %}
        <b>Signals:</b>
        <ul style="margin:8px 0 0 18px;">
          {% for r in header_risk.reasons %}
            <li>
              <b>[{{ r.severity }}]</b> {{ r.message }}
              <div style="font-size:12px; opacity:0.85; margin-top:4px;">
                {% for k,v in r.evidence.items() %}
                  <div>• {{ k }}: {{ v }}</div>
                {% endfor %}
              </div>
            </li>
          {% endfor %}
        </ul>
        {% else %}
        <div>No header signals found (or headers not provided).</div>
        {% endif %}
    </div>
    {% endif %}

    {% if prediction_label %}
    <div class="result">
        ▸ Prediction: {{ prediction_label }} <br>
        ▸ Probability Index: {{ prediction_prob | round(3) }}
    </div>
    {% endif %}
</div>

</body>
</html>
"""


# -----------------------------
# ROUTES (correct, complete)
# Keep ONLY these two routes in your file.
# -----------------------------

@app.route('/', methods=['GET'])
def home():
    # Initial page load: nothing predicted yet
    return render_template_string(
        HOME_HTML,
        prediction_label=None,
        prediction_prob=None,
        header_risk=None
    )


@app.route('/predict', methods=['POST'])
def predict():
    try:
        # Accept form or JSON
        if request.is_json:
            data = request.get_json(silent=True) or {}
            email_text = data.get('email_text', '')
        else:
            email_text = request.form.get('email_text', '')

        # ML prediction
        features = vectorizer.transform([email_text]).toarray().astype(np.float32)
        prediction_prob = float(model.predict(features)[0][0])
        prediction_label = "Phishing" if prediction_prob >= 0.5 else "Safe"

        # Header-based explainability (no retraining)
        header_risk = compute_header_risk_from_raw_text(
            email_text,
            internal_domains=["company.com"],   # <-- EDIT
            exec_names=["Jane CEO", "John CFO"] # <-- OPTIONAL
        )

        # Return JSON for API callers
        if request.is_json:
            return jsonify({
                "prediction_label": prediction_label,
                "prediction_prob": prediction_prob,
                "header_risk": header_risk
            })

        # Render HTML for browser form submits
        return render_template_string(
            HOME_HTML,
            prediction_label=prediction_label,
            prediction_prob=prediction_prob,
            header_risk=header_risk
        )

    except Exception as e:
        if request.is_json:
            return jsonify({'error': str(e)}), 500

        # Browser-friendly error display
        return render_template_string(
            HOME_HTML,
            prediction_label="Error",
            prediction_prob=0.0,
            header_risk={
                "tier": "ERROR",
                "header_risk_score": 0.0,
                "reasons": [{"id": "SERVER_ERROR", "severity": "HIGH", "message": str(e), "evidence": {}}],
                "extracted": {}
            }
        ), 500


if __name__ == '__main__':
    app.run(debug=True, use_reloader=False)

 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [08/Apr/2026 04:14:31] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [08/Apr/2026 04:14:32] "GET /docs HTTP/1.1" 200 -
127.0.0.1 - - [08/Apr/2026 04:14:33] "GET /templates/flashpython.png HTTP/1.1" 404 -
127.0.0.1 - - [08/Apr/2026 04:14:33] "GET /templates/picklepython.png HTTP/1.1" 404 -
127.0.0.1 - - [08/Apr/2026 04:14:33] "GET /templates/github.png HTTP/1.1" 404 -
127.0.0.1 - - [08/Apr/2026 04:14:33] "GET /templates/htmlcsslogo.png HTTP/1.1" 404 -
127.0.0.1 - - [08/Apr/2026 04:14:36] "GET /about HTTP/1.1" 200 -
127.0.0.1 - - [08/Apr/2026 04:14:37] "GET / HTTP/1.1" 200 -
